In [1]:
# %%
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
device_ids = [0]

import gcsfs
import jax
import jax.numpy as jnp
import numpy as np
import pickle
import xarray
import neuralgcm
import pandas as pd
import matplotlib.pyplot as plt

print(jax.devices())
gcs = gcsfs.GCSFileSystem(token='anon')


[cuda(id=0)]


In [ ]:

# %%
# 辅助函数: 堆叠和拆分字典列表
def stack_dicts(dicts_list):
    """Stacks a list of dictionaries into a single dictionary with a batch dimension."""
    return jax.tree_util.tree_map(lambda *args: jnp.stack(args), *dicts_list)

def unstack_dicts(batched_dict, batch_size):
    """Splits a batched dictionary back into a list of dictionaries."""
    return [
        jax.tree_util.tree_map(lambda x: x[i], batched_dict)
        for i in range(batch_size)
    ]

# %%
class NeuralGCMInference:
    def __init__(self, checkpoint_path, inner_steps=1, outer_steps=24):
        """
        初始化 NeuralGCM 推理对象.
        
        Args:
            checkpoint_path: 模型权重文件路径.
            inner_steps: 内部步数 (例如 1 小时).
            outer_steps: 外部步数 (预测总长度).
        """
        self.inner_steps = inner_steps
        self.outer_steps = outer_steps
        self.timedelta = np.timedelta64(1, 'h') * inner_steps
        self.times = (np.arange(outer_steps) * inner_steps)
        
        print(f"Loading model from {checkpoint_path}...")
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
            
        self.model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
        self.model_with_physics = self.model.with_physics_core_output(enable=True)
        
        print(f"Model resolution: {360/self.model.data_coords.horizontal.longitude_nodes:.2f} deg")
        
        # 编译 JAX 函数
        print("Compiling JAX functions...")
        self._compile_functions()
        print("Functions compiled.")

    def _compile_functions(self):
        """编译 vmap 版本的 encode 和 unroll 函数."""
        
        # 1. Batched Encode
        self.batched_encode_fn = jax.jit(jax.vmap(self.model.encode))
        
        # 2. Batched Unroll
        def unroll_wrapper(state, forcings):
            return self.model_with_physics.unroll(
                state,
                forcings,
                steps=self.outer_steps,
                timedelta=self.timedelta,
                start_with_input=True
            )
        
        self.batched_unroll_fn = jax.jit(jax.vmap(unroll_wrapper))

    def _prepare_batch_inputs(self, datasets_list):
        """准备批量输入数据."""
        inputs_list = []
        input_forcings_list = []
        all_forcings_list = []
        target_datasets = []

        print(f"Preparing inputs for {len(datasets_list)} datasets...")
        
        for i, ds in enumerate(datasets_list):
            # 1. Inputs (t=0)
            inputs_list.append(self.model.inputs_from_xarray(ds.isel(time=0)))
            input_forcings_list.append(self.model.forcings_from_xarray(ds.isel(time=0)))
            
            # 2. Forcings (Persistence)
            # 根据需求，如果以后要改 Dynamic，这里也要改
            all_forcings_list.append(self.model.forcings_from_xarray(ds.head(time=1)))
            
            # 3. Target
            target_trajectory = self.model.inputs_from_xarray(
                ds
                .thin(time=(self.inner_steps // 1)) 
                .isel(time=slice(self.outer_steps))
            )
            target_datasets.append(self.model.data_to_xarray(target_trajectory, times=self.times))

        batched_inputs = stack_dicts(inputs_list)
        batched_input_forcings = stack_dicts(input_forcings_list)
        batched_all_forcings = stack_dicts(all_forcings_list)
        
        rng_key = jax.random.key(42)
        batched_rng_keys = jax.random.split(rng_key, len(datasets_list))
        
        return batched_inputs, batched_input_forcings, batched_all_forcings, batched_rng_keys, target_datasets

    def forward(self, datasets_list):
        """
        执行推理流程.
        
        Args:
            datasets_list: 已读取并切片好的 xarray.Dataset 列表.
            
        Returns:
            predictions_list: 预测结果列表 (xarray.Dataset).
            target_datasets: 真实值列表 (xarray.Dataset).
        """
        print("开始数据准备")
        # 1. 数据准备
        (batched_inputs, batched_input_forcings, batched_all_forcings, 
         batched_rng_keys, target_datasets) = self._prepare_batch_inputs(datasets_list)
        
        print("开始encoder")
        # 2. 编码 (Encode)
        print("Running batched encode...")
        batched_initial_state = self.batched_encode_fn(
            batched_inputs, batched_input_forcings, batched_rng_keys
        )
        
        print("开始unroll")
        # 3. 预测 (Unroll)
        print("Running batched unroll...")
        _, batched_predictions_with_physics = self.batched_unroll_fn(
            batched_initial_state, 
            batched_all_forcings
        )
        
        print("开始后处理")
        # 4. 后处理 (Post-process)
        print("Processing outputs...")
        raw_predictions_list = unstack_dicts(batched_predictions_with_physics, len(datasets_list))
        
        final_results = []
        for i, (raw_pred, target_ds) in enumerate(zip(raw_predictions_list, target_datasets)):
            # 转换回 xarray (同时拿到 physics core 和 full prediction)
            pred_ds, pred_phy_ds = self.model_with_physics.data_to_xarray_with_physics(
                raw_pred, times=self.times
            )
            
            # 拼接
            combined = xarray.concat([target_ds, pred_phy_ds, pred_ds], 'model')
            combined.coords['model'] = ['ERA5', 'Physics_Core', 'NeuralGCM']
            final_results.append(combined)
            
        return final_results



In [2]:
# %%
model_name = 'neuralgcm_04_30_2024_neural_gcm_dynamic_forcing_deterministic_1_4_deg.pkl'
checkpoint_path = f'/nfs/gpu_homes/gpu09/home/zhangjing/Code/NeuralGCM/pkl/{model_name}'

# 1. 初始化模型
ngcm = NeuralGCMInference(checkpoint_path, inner_steps=1, outer_steps=24)

# 2. 准备数据文件列表
file_configs = [
    ('/nfs/samba/数据聚变/气象数据/ERA5_Global_37Level_01Degree/Uncompressed_Regridded_and_Filled/2022/04/01/20220401.nc', '2022-04-01 00:00:00'),
    ('/nfs/samba/数据聚变/气象数据/ERA5_Global_37Level_01Degree/Uncompressed_Regridded_and_Filled/2021/01/01/20210101.nc', '2021-01-01 00:00:00'),
    # ('/nfs/samba/数据聚变/气象数据/ERA5_Global_37Level_01Degree/Uncompressed_Regridded_and_Filled/2021/01/05/20210105.nc', '2021-01-05 00:00:00'),
]

print("正在读取文件列表...")
loaded_datasets = []
start_times_list = []
for pth, start_time in file_configs:
    ds = xarray.open_dataset(pth, decode_timedelta=False)
    loaded_datasets.append(ds)
    start_times_list.append(start_time)



Loading model from /nfs/gpu_homes/gpu09/home/zhangjing/Code/NeuralGCM/pkl/neuralgcm_04_30_2024_neural_gcm_dynamic_forcing_deterministic_1_4_deg.pkl...
Model resolution: 1.41 deg
Compiling JAX functions...
Functions compiled.
正在读取文件列表...


In [ ]:
# 3. 执行推理 (Forward)
results_list = ngcm.forward(loaded_datasets)
print(f"Inference complete. Got {len(results_list)} results.")


开始数据准备
Preparing inputs for 2 datasets...


In [ ]:


# %%
# 这里只演示画第一个结果
if len(results_list) > 0:
    plot_index = 0
    print(f"Plotting result for item {plot_index}...")
    
    combined_ds = results_list[plot_index]
    
    target = combined_ds.sel(model="ERA5")
    baseline = combined_ds.sel(model="NeuralGCM")
    baseline_phy = combined_ds.sel(model="Physics_Core")
    
    diff = target - baseline
    diff_phy = target - baseline_phy
    
    var_map = {
        "z": "geopotential", "s": "specific_humidity", "t": "temperature",
        "u": "u_component_of_wind", "v": "v_component_of_wind",
    }
    levels_to_plot = [50, 500, 850, 1000]
    
    # 预计算 RMSE 并加载到内存 (优化速度)
    rmse_ds = (diff ** 2).mean(dim=['longitude', 'latitude'], skipna=True) ** 0.5
    rmse_phy_ds = (diff_phy ** 2).mean(dim=['longitude', 'latitude'], skipna=True) ** 0.5
    rmse_ds.load()
    rmse_phy_ds.load()
    
    fig, axes = plt.subplots(len(var_map), len(levels_to_plot), figsize=(18, 16), constrained_layout=True)
    
    for row_idx, (label, var_name) in enumerate(var_map.items()):
        for col_idx, level_value in enumerate(levels_to_plot):
            ax = axes[row_idx, col_idx]
            
            rmse_ds[var_name].sel(level=level_value, method="nearest").plot(ax=ax, label="NeuralGCM")
            rmse_phy_ds[var_name].sel(level=level_value, method="nearest").plot(ax=ax, label="Physics_Core", color='red')
            
            ax.set_title(f"{label.upper()} {level_value}hPa")
            ax.set_xlabel("Hour")
            ax.set_ylabel("RMSE")
            ax.legend()
            
    plt.show()


In [ ]:
# %%
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
device_ids = [0]

import gcsfs
import jax
import jax.numpy as jnp
import numpy as np
import pickle
import xarray
import neuralgcm
import pandas as pd
import matplotlib.pyplot as plt
import torch  # 新增 torch 导入
from jax import dlpack as jax_dlpack
from torch.utils import dlpack as torch_dlpack
import time # 用于计时

# 确保 JAX 使用 GPU
print(f"JAX Devices: {jax.devices()}")
print(f"Torch Devices: {torch.cuda.device_count()}")
gcs = gcsfs.GCSFileSystem(token='anon')

# %%
# 辅助函数: 堆叠和拆分字典列表
def stack_dicts(dicts_list):
    """Stacks a list of dictionaries into a single dictionary with a batch dimension."""
    # jnp.stack automatically moves data to the default device (GPU if available)
    return jax.tree_util.tree_map(lambda *args: jnp.stack(args), *dicts_list)

def unstack_dicts(batched_dict, batch_size):
    """Splits a batched dictionary back into a list of dictionaries."""
    return [
        jax.tree_util.tree_map(lambda x: x[i], batched_dict)
        for i in range(batch_size)
    ]

# %%
# 核心堆叠函数 (普通 JAX 函数，将被集成到 JIT 块中)
def _core_stack_and_transpose(data_dict, level_indices):
    """
    Core stacking logic to be used INSIDE a JIT function.
    Args:
        data_dict: Dictionary of JAX arrays (Batch, Time, Level, Lat, Lon)
        level_indices: Array of indices to select from Level dimension
    """
    
    # 硬编码变量列表，确保 JIT 稳定性
    # 如果模型变量名改变，这里需要同步修改
    # 恢复为只包含基础变量 (5个)，去除云变量，以恢复20个通道的输出 (5变量 * 4层)
    multi_level_vars = ['geopotential', 'temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind']
    # 如果后续需要云变量，可以取消注释：
    # multi_level_vars += ['specific_cloud_ice_water_content', 'specific_cloud_liquid_water_content']
    
    # Single-level vars: 
    possible_single_vars = [
        '2m_temperature', 
        '10m_u_component_of_wind', 
        '10m_v_component_of_wind', 
        'mean_sea_level_pressure',
        'sea_surface_temperature', 
        'sea_ice_cover'
    ]
    
    # Helper to extract and stack one set of vars
    def process_subset(prefix=''):
        channels = []
        
        # 1. Process Multi-level
        multi_arrays = []
        for v in multi_level_vars:
            key = prefix + v
            # JIT time check: key existence is static
            if key in data_dict:
                multi_arrays.append(data_dict[key])
        
        if multi_arrays:
            stacked = jnp.stack(multi_arrays, axis=0) # (Vars, Batch, Time, Level, Lat, Lon)
            selected = stacked[:, :, :, level_indices, :, :]
            permuted = jnp.transpose(selected, (0, 3, 1, 2, 4, 5)) # (Var, Level, Batch, Time, Lat, Lon)
            flat = permuted.reshape(-1, *permuted.shape[2:]) # (Level*Var, Batch, Time, Lat, Lon)
            channels.append(flat)
            
        # 2. Process Single-level
        single_arrays = []
        for v in possible_single_vars:
            key = prefix + v
            if key in data_dict:
                arr = data_dict[key]
                # Handle possible dimensions
                # Sometimes single level vars have a singleton level dim (Batch, Time, 1, Lat, Lon)
                if arr.ndim == 5 and arr.shape[2] == 1: 
                    arr = arr.squeeze(axis=2)
                
                # Ensure it's (Batch, Time, Lat, Lon)
                if arr.ndim == 4:
                    single_arrays.append(arr)
        
        if single_arrays:
            # Stack along new dim -> (Vars, Batch, Time, Lat, Lon)
            stacked_single = jnp.stack(single_arrays, axis=0)
            channels.append(stacked_single)
            
        if not channels:
            return None
            
        # Concatenate all (Multi + Single)
        final_stack = jnp.concatenate(channels, axis=0) # (TotalCh, Batch, Time, Lat, Lon)
        
        # Transpose to (Batch, TotalCh, Lat, Lon, Time)
        return jnp.transpose(final_stack, (1, 0, 3, 4, 2)) 

    # Process Full
    tensor_full = process_subset(prefix='')
    
    # Process Physics Core (prefix '_physics_')
    tensor_phy = process_subset(prefix='_physics_')
    
    return tensor_full, tensor_phy


class NeuralGCMInference:
    def __init__(self, checkpoint_path, inner_steps=1, outer_steps=24):
        """
        初始化 NeuralGCM 推理对象.
        """
        self.inner_steps = inner_steps
        self.outer_steps = outer_steps
        self.timedelta = np.timedelta64(1, 'h') * inner_steps
        self.times = (np.arange(outer_steps) * inner_steps)
        
        print(f"Loading model from {checkpoint_path}...")
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
            
        self.model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
        self.model_with_physics = self.model.with_physics_core_output(enable=True)
        
        # 获取所有气压层数值
        self.all_levels = self.model.data_coords.vertical.centers
        print(f"Model levels ({len(self.all_levels)}): {self.all_levels}")
        
        # 编译 JAX 函数
        print("Compiling JAX functions (Fused)...")
        self._compile_fused_functions()
        print("Functions compiled.")
        
        # 预热 & 探测 Keys
        print("Warming up and detecting output keys...")
        self._warmup_and_debug_keys()
        print("Warmup complete.")

    def _compile_fused_functions(self):
        """
        编译融合了 Encode + Unroll + Stack 的超大 JIT 函数。
        """
        
        # 1. 基础 unroll wrapper
        def unroll_op(state, forcings):
            return self.model_with_physics.unroll(
                state,
                forcings,
                steps=self.outer_steps,
                timedelta=self.timedelta,
                start_with_input=True
            )
        
        # 2. 融合函数
        def forward_and_stack(inputs, input_forcings, rng_keys, all_forcings, level_indices):
            initial_state = jax.vmap(self.model.encode)(inputs, input_forcings, rng_keys)
            _, predictions_dict = jax.vmap(unroll_op)(initial_state, all_forcings)
            
            t_full, t_phy = _core_stack_and_transpose(predictions_dict, level_indices)
            return t_full, t_phy

        self.fused_forward_fn = jax.jit(forward_and_stack)
        
        # Target 的 stack
        def stack_target(target_dict, level_indices):
            t_full, _ = _core_stack_and_transpose(target_dict, level_indices)
            return t_full
            
        self.stack_target_fn = jax.jit(stack_target)
        
        # DEBUG Helper
        def debug_step_fn(inputs, input_forcings, rng_key, all_forcings):
            state = self.model.encode(inputs, input_forcings, rng_key)
            _, preds = self.model_with_physics.unroll(
                state, all_forcings, steps=1, timedelta=self.timedelta, start_with_input=True
            )
            return preds
        
        self._debug_step_fn = debug_step_fn

    def _prepare_batch_inputs(self, datasets_list):
        """准备批量输入数据，并确保它们被移动到 GPU."""
        inputs_list = []
        input_forcings_list = []
        all_forcings_list = []
        target_datasets = []

        print(f"Preparing inputs for {len(datasets_list)} datasets...")
        
        for i, ds in enumerate(datasets_list):
            # inputs_from_xarray 返回 numpy 数组 (CPU)
            inputs_list.append(self.model.inputs_from_xarray(ds.isel(time=0)))
            input_forcings_list.append(self.model.forcings_from_xarray(ds.isel(time=0)))
            all_forcings_list.append(self.model.forcings_from_xarray(ds.head(time=1)))
            
            target_trajectory = self.model.inputs_from_xarray(
                ds
                .thin(time=(self.inner_steps // 1)) 
                .isel(time=slice(self.outer_steps))
            )
            target_datasets.append(target_trajectory)

        # stack_dicts 使用 jnp.stack。JAX 会自动将数据移动到默认设备 (通常是 GPU 0)
        # 但为了绝对保险，我们显式 device_put
        
        batched_inputs = stack_dicts(inputs_list)
        batched_input_forcings = stack_dicts(input_forcings_list)
        batched_all_forcings = stack_dicts(all_forcings_list)
        batched_targets = stack_dicts(target_datasets)
        
        # Explicitly move to GPU (if not already)
        # 这一步会发生数据传输 (CPU -> GPU)，这是不可避免的，但只发生一次
        batched_inputs = jax.device_put(batched_inputs)
        batched_input_forcings = jax.device_put(batched_input_forcings)
        batched_all_forcings = jax.device_put(batched_all_forcings)
        batched_targets = jax.device_put(batched_targets)
        
        rng_key = jax.random.key(42)
        batched_rng_keys = jax.random.split(rng_key, len(datasets_list))
        
        return batched_inputs, batched_input_forcings, batched_all_forcings, batched_rng_keys, batched_targets

    def _warmup_and_debug_keys(self):
        """Debug Output Keys."""
        try:
            # Note: Output variable keys depend on the specific model config.
            # We skip actual warmup run to avoid messing up with different input shapes.
            pass
        except Exception as e:
            print(f"Warmup warning: {e}")

    def forward(self, datasets_list, target_levels=None, include_era5_label=True):
        """
        执行推理流程并返回 Tensor (GPU).
        """
        t0 = time.time()
        # 1. 数据准备 (包含移动到 GPU)
        (batched_inputs, batched_input_forcings, batched_all_forcings, 
         batched_rng_keys, batched_targets) = self._prepare_batch_inputs(datasets_list)
        
        # 确认输入已在 GPU
        # print(f"Debug: Input device: {list(batched_inputs.values())[0].device}")
        
        t1 = time.time()
        print(f"Data Prep Time: {t1-t0:.4f}s")
        
        # 2. 准备 Level Indices
        if target_levels is not None:
            level_indices = [np.argmin(np.abs(self.all_levels - l)) for l in target_levels]
        else:
            level_indices = list(range(len(self.all_levels)))
        level_indices_arr = jnp.array(level_indices, dtype=jnp.int32)
        
        # DEBUG: Peek keys
        if not hasattr(self, '_keys_printed'):
            print("--- Debug: Inspecting Model Output Keys ---")
            try:
                single_in = jax.tree_util.tree_map(lambda x: x[0], batched_inputs)
                single_in_f = jax.tree_util.tree_map(lambda x: x[0], batched_input_forcings)
                single_all_f = jax.tree_util.tree_map(lambda x: x[0], batched_all_forcings)
                single_rng = batched_rng_keys[0]
                preds = self._debug_step_fn(single_in, single_in_f, single_rng, single_all_f)
                keys = sorted(list(preds.keys()))
                print(f"Available Keys: {keys}")
                self._keys_printed = True
            except Exception as e:
                print(f"Debug run failed: {e}")
            print("-------------------------------------------")

        # 3. 融合推理 (Fused Forward)
        print("Running Fused Inference (Encode->Unroll->Stack)...")
        jax_full, jax_phy = self.fused_forward_fn(
            batched_inputs, batched_input_forcings, batched_rng_keys, 
            batched_all_forcings, level_indices_arr
        )
        
        if jax_full is not None:
            jax.block_until_ready(jax_full)
        t2 = time.time()
        print(f"True Inference Time (GPU): {t2-t1:.4f}s")
        
        # 4. Target Stacking
        jax_target = None
        if include_era5_label:
            jax_target = self.stack_target_fn(batched_targets, level_indices_arr)
            if jax_target is not None:
                jax.block_until_ready(jax_target)
            
        print("Inference done. Converting DLPack to Torch...")
        
        # 5. DLPack Zero-Copy (现在输入肯定在 GPU，转换也应该保持在 GPU)
        t_full = torch_dlpack.from_dlpack(jax_dlpack.to_dlpack(jax_full)) if jax_full is not None else None
        t_phy = torch_dlpack.from_dlpack(jax_dlpack.to_dlpack(jax_phy)) if jax_phy is not None else None
        t_target = torch_dlpack.from_dlpack(jax_dlpack.to_dlpack(jax_target)) if jax_target is not None else None
        
        t3 = time.time()
        print(f"Conversion Time: {t3-t2:.4f}s")
        print(f"Total Forward Time: {t3-t0:.4f}s")
        
        return t_full, t_phy, t_target

# %%
def calculate_rmse_and_plot(t_pred, t_phy, t_target, target_levels, output_dir="plots"):
    """
    计算 RMSE 并绘制结果 (Grid Subplots: Rows=Variables, Cols=Levels)
    t_pred, t_phy, t_target shape: (Batch, Channels, Lat, Lon, Time)
    Channels order: Level1_Var1, Level1_Var2... Level2_Var1...
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # 变量列表 (必须与 _core_stack_and_transpose 一致)
    multi_level_vars = ['geopotential', 'temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind']
    num_vars = len(multi_level_vars)
    num_levels = len(target_levels)
    
    # 确保 Channel 数匹配
    expected_channels = num_vars * num_levels
    if t_pred.shape[1] < expected_channels:
        print(f"Warning: Prediction channels ({t_pred.shape[1]}) < Expected ({expected_channels}). Plotting may be incomplete.")
    
    # 时间步
    time_steps = t_pred.shape[-1]
    times = np.arange(time_steps)
    
    print(f"Plotting RMSE Grid: {num_vars} Rows (Vars) x {num_levels} Cols (Levels)...")
    
    # 创建大图 Grid
    # figsize: width, height. 每列宽约4，每行高约3
    fig, axes = plt.subplots(nrows=num_vars, ncols=num_levels, figsize=(4*num_levels, 3*num_vars), sharex=True)
    
    # 如果只有一行或一列，axes 可能不是二维数组，统一处理
    if num_vars == 1 and num_levels == 1:
        axes = np.array([[axes]])
    elif num_vars == 1:
        axes = axes.reshape(1, -1)
    elif num_levels == 1:
        axes = axes.reshape(-1, 1)
        
    # 遍历每个变量 (Rows)
    for v_idx, var_name in enumerate(multi_level_vars):
        # 遍历每个高度 (Cols)
        for l_idx, level in enumerate(target_levels):
            ax = axes[v_idx, l_idx]
            
            # 计算当前变量在 Channel 中的索引
            # 顺序: Var1 (all levels), Var2 (all levels)...
            # 所以: Index = v_idx * num_levels + l_idx
            ch_idx = v_idx * num_levels + l_idx
            
            if ch_idx >= t_pred.shape[1]:
                ax.text(0.5, 0.5, 'Missing Data', ha='center', va='center')
                continue
                
            # 提取数据 (Batch, Lat, Lon, Time)
            pred_slice = t_pred[:, ch_idx, :, :, :]
            phy_slice = t_phy[:, ch_idx, :, :, :]
            target_slice = t_target[:, ch_idx, :, :, :]
            
            # 计算 RMSE (Batch, Lat, Lon, Time) -> 平均 Batch, Lat, Lon -> (Time,)
            # Full Model RMSE
            mse_full = torch.mean((pred_slice - target_slice) ** 2, dim=(0, 1, 2))
            rmse_full = torch.sqrt(mse_full).cpu().numpy()
            
            # Physics Core RMSE
            mse_phy = torch.mean((phy_slice - target_slice) ** 2, dim=(0, 1, 2))
            rmse_phy = torch.sqrt(mse_phy).cpu().numpy()
            
            # 绘图
            ax.plot(times, rmse_full, label='NeuralGCM', color='blue', marker='o', markersize=2)
            ax.plot(times, rmse_phy, label='Physics Core', color='red', linestyle='--', marker='x', markersize=2)
            
            # 标题仅在第一行显示 Level，或者每个子图都显示详细标题
            # 这里选择每个子图显示简略标题
            ax.set_title(f'{var_name}\n@{level} hPa', fontsize=10)
            
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # 仅第一列显示 Y 轴标签
            if l_idx == 0:
                ax.set_ylabel('RMSE')
            
            # 仅最后一行显示 Legend 和 X 轴标签
            if v_idx == num_vars - 1:
                ax.set_xlabel('Time Steps (h)')
            
            # Legend 放在每张图里可能会太挤，建议放在第一个图或者统一放在外面
            # 这里选择只在第一个子图显示图例，或者根据需要调整
            if v_idx == 0 and l_idx == num_levels - 1:
                ax.legend(loc='upper right', fontsize='small')

    plt.tight_layout()
    plt.show() # 在 Notebook 中直接显示

# %% [markdown]
# ## 使用示例 (Usage Example)

# %%
model_name = 'neuralgcm_04_30_2024_neural_gcm_dynamic_forcing_deterministic_1_4_deg.pkl'
checkpoint_path = f'/nfs/gpu_homes/gpu09/home/zhangjing/Code/NeuralGCM/pkl/{model_name}'

# 1. 初始化模型
ngcm = NeuralGCMInference(checkpoint_path, inner_steps=1, outer_steps=24)

# 2. 准备数据文件列表
file_configs = [
    ('/nfs/samba/数据聚变/气象数据/ERA5_Global_37Level_01Degree/Uncompressed_Regridded_and_Filled/2022/04/01/20220401.nc', '2022-04-01 00:00:00'),
    ('/nfs/samba/数据聚变/气象数据/ERA5_Global_37Level_01Degree/Uncompressed_Regridded_and_Filled/2021/01/01/20210101.nc', '2021-01-01 00:00:00'),
]

print("Reading datasets...")
loaded_datasets = []
for pth, start_time in file_configs:
    try:
        ds = xarray.open_dataset(pth, decode_timedelta=False)
        start_dt = pd.Timestamp(start_time)
        end_dt = start_dt + pd.Timedelta(hours=ngcm.outer_steps + 1)
        ds_sel = ds.sel(time=slice(start_dt, end_dt))
        if ds_sel.time.size < 1: continue
        loaded_datasets.append(ds_sel)
    except: continue

# 3. 执行推理 (Forward) -> 返回 Tensor (GPU)
target_levels = [50, 500, 850, 1000] # 从低到高排列
print(f"Target Levels: {target_levels}")

t_pred, t_phy, t_target = ngcm.forward(
    loaded_datasets, 
    target_levels=target_levels, 
    include_era5_label=True
)

print("\nOutput Shapes:")
if t_pred is not None: print(f"NeuralGCM Pred: {t_pred.shape}")
if t_phy is not None: print(f"Physics Core:   {t_phy.shape}")
if t_target is not None: print(f"ERA5 Target:    {t_target.shape}")

# 验证数据是否在 GPU 上
if t_pred is not None: print(f"Tensor Device: {t_pred.device}")

# 4. 计算 RMSE 并绘图
if t_pred is not None and t_phy is not None and t_target is not None:
    calculate_rmse_and_plot(t_pred, t_phy, t_target, target_levels)
